# 04 — Ablations A–E

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/04_ablations.ipynb)

Les cinq variantes de docs/03-classifieur.md §4 (`configs/ablations/`), entraînées puis
évaluées l'une après l'autre ; tableau final dans `reports/phase1.md`. Les runs terminés
sont enregistrés dans `reports/ablations.json` : une session interrompue reprend à
l'ablation suivante (retirer une entrée pour la relancer). Compter plusieurs heures par
ablation sur un T4.

Sur Colab : *Exécution → Modifier le type d'exécution → GPU (T4)*.

## Paramètres

In [3]:
ABLATIONS = ["A_baseline", "B_strong_aug", "C_background", "D_ccmt", "E_convnext"]
SET = ["augment.background.enabled=false"] # surcharges appliquées à toutes les ablations
SUBSET = 0
LIMIT_BATCHES = 0
DEVICE = None
DATASETS = ["index", "plantvillage", "ccmt", "plantdoc"]

## Environnement

In [4]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

Mounted at /content/drive
/content/drive/MyDrive/SerreIA/runs/runs → runs
/content/drive/MyDrive/SerreIA/runs/reports → reports
/content/drive/MyDrive/SerreIA/runs/mlruns/colab → mlruns/colab
$ /content/SerreIA/tools/get_datasets.sh unpack /content/drive/MyDrive/SerreIA/data index
$ /content/SerreIA/tools/get_datasets.sh unpack /content/drive/MyDrive/SerreIA/data plantvillage
$ /content/SerreIA/tools/get_datasets.sh unpack /content/drive/MyDrive/SerreIA/data ccmt
$ /content/SerreIA/tools/get_datasets.sh unpack /content/drive/MyDrive/SerreIA/data plantdoc


In [5]:
# Entraînement, évaluation et Grad-CAM tournent sur le GPU, augmentations comprises.
import torch

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
elif COLAB:
    raise RuntimeError("pas de GPU : Exécution → Modifier le type d'exécution → GPU (T4)")
else:
    print("pas de GPU : tout tournera sur CPU (lent)")

GPU : NVIDIA A100-SXM4-80GB


In [6]:
def ensure_backgrounds(cfg):
    """Fonds de serre requis par le remplacement de fond (docs/03 §3.2)."""
    if not cfg["augment"]["background"]["enabled"]:
        return
    try:
        drive_data.ensure(["backgrounds"], DRIVE_DIR if COLAB else None)
    except RuntimeError as e:
        raise RuntimeError(
            f"{e}\nRemplir data/backgrounds/ puis `tools/get_datasets.sh push backgrounds`, "
            'ou SET = ["augment.background.enabled=false"].'
        ) from None

## Entraînement et évaluation de chaque ablation

In [7]:
import torch

from vision.eval.ablation_report import load_registry, save_registry
from vision.eval.cross_domain_eval import REPORTS_DIR, log_to_mlflow, run_eval
from vision.train.config import apply_overrides, load_config
from vision.train.train_classifier import pick_device, run

device = pick_device(DEVICE)
registry = load_registry()
for name in ABLATIONS:
    done = registry.get(name)
    if done and (REPORTS_DIR / done / "metrics.json").exists():
        print(f"{name} : déjà fait ({done})")
        continue
    cfg = apply_overrides(
        load_config(ROOT / "configs" / "ablations" / f"{name}.yaml"), ENV_SET + SET
    )
    ensure_backgrounds(cfg)
    print(f"=== {name} ===")
    with drive_data.PeriodicSync(drive_data.OUTPUTS, DRIVE_DIR):
        run_id = run(cfg, device, SUBSET, LIMIT_BATCHES).name
        with torch.inference_mode():
            payload = run_eval(ROOT / "runs" / run_id / "best.pt", device, SUBSET)
        log_to_mlflow(run_id, payload, REPORTS_DIR / run_id)
        registry[name] = run_id
        save_registry(registry)

INFO NumExpr defaulting to 12 threads.


A_baseline : déjà fait (84e830ff98c24ae09688fdb87fc529f5)
B_strong_aug : déjà fait (f780dce15d7a4af3b9f429ae9820a369)
C_background : déjà fait (4c9cf2d5dffd4eb0839704124cdaf569)
=== D_ccmt ===


INFO groupes contradictoires : 715 images écartées {'ccmt': 612, 'plantvillage': 64, 'plantdoc': 39}
INFO train 62993 images, val 7878 images
INFO Loading pretrained weights from Hugging Face hub (timm/efficientnet_b0.ra_in1k)
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
INFO HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/timm/efficientnet_b0.ra_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO HTTP Request: GET https://huggingface.co/api/models/timm/efficientnet_b0.ra_in1k/xet-read-token/1b5383e5f79cc0f7fc067e372f8f26a5fa73f26a "HTTP/1.1 200 OK"


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

...

runs → /content/drive/MyDrive/SerreIA/runs/runs
reports → /content/drive/MyDrive/SerreIA/runs/reports
mlruns/colab → /content/drive/MyDrive/SerreIA/runs/mlruns/colab


INFO epoch 16  loss 1.297  val sante/f1_macro 0.9938  classe f1 0.9768
INFO epoch 17  loss 1.282  val sante/f1_macro 0.9934  classe f1 0.9807
INFO epoch 18  loss 1.271  val sante/f1_macro 0.9949  classe f1 0.9789
INFO epoch 19  loss 1.265  val sante/f1_macro 0.9952  classe f1 0.9815
INFO epoch 20  loss 1.257  val sante/f1_macro 0.9957  classe f1 0.9817
INFO epoch 21  loss 1.250  val sante/f1_macro 0.9954  classe f1 0.9804
INFO epoch 22  loss 1.244  val sante/f1_macro 0.9956  classe f1 0.9829
INFO epoch 23  loss 1.240  val sante/f1_macro 0.9959  classe f1 0.9825


runs → /content/drive/MyDrive/SerreIA/runs/runs
reports → /content/drive/MyDrive/SerreIA/runs/reports
mlruns/colab → /content/drive/MyDrive/SerreIA/runs/mlruns/colab


INFO epoch 24  loss 1.236  val sante/f1_macro 0.9956  classe f1 0.9838
INFO epoch 25  loss 1.234  val sante/f1_macro 0.9956  classe f1 0.9833
INFO epoch 26  loss 1.231  val sante/f1_macro 0.9956  classe f1 0.9835
INFO epoch 27  loss 1.232  val sante/f1_macro 0.9956  classe f1 0.9838
INFO run 96f8b511d38c414890b750f2123e3fa7, meilleur sante/f1_macro = 0.9959 -> /content/SerreIA/runs/96f8b511d38c414890b750f2123e3fa7
INFO groupes contradictoires : 715 images écartées {'ccmt': 612, 'plantvillage': 64, 'plantdoc': 39}
INFO test : 7884 images
INFO test_xdomain : 8665 images
INFO rapport écrit dans /content/SerreIA/reports/96f8b511d38c414890b750f2123e3fa7


runs → /content/drive/MyDrive/SerreIA/runs/runs
reports → /content/drive/MyDrive/SerreIA/runs/reports
mlruns/colab → /content/drive/MyDrive/SerreIA/runs/mlruns/colab


## Rapport `reports/phase1.md`

In [8]:
from IPython.display import Markdown, display

from vision.eval.ablation_report import write_phase1_report

report = write_phase1_report({k: v for k, v in registry.items() if k in ABLATIONS})
drive_data.sync_all(DRIVE_DIR)
display(Markdown(report.read_text(encoding="utf-8")))

runs → /content/drive/MyDrive/SerreIA/runs/runs
reports → /content/drive/MyDrive/SerreIA/runs/reports
mlruns/colab → /content/drive/MyDrive/SerreIA/runs/mlruns/colab


# Phase 1 — ablations du classifieur

Généré le 13/10/2026 par `python -m vision.eval.ablation_report` (voir [docs/03-classifieur.md](../docs/03-classifieur.md) §4).

| ablation | run | F1 sante (test) | F1 classe (test) | rappel anormal (PlantDoc) | F1 sante (PlantDoc) | AUROC sante (PlantDoc) | F1 classe (PlantDoc) | top-3 classe (PlantDoc) | ECE sante (PlantDoc) |
|---|---|---|---|---|---|---|---|---|---|
| A_baseline | `84e830ff` | 1.000 | 0.998 | 0.869 | 0.705 | 0.820 | 0.188 | 0.390 | 0.148 |
| B_strong_aug | `f780dce1` | 1.000 | 0.993 | 0.960 | 0.734 | 0.870 | 0.286 | 0.530 | 0.115 |
| C_background | `4c9cf2d5` | 1.000 | 0.992 | 0.973 | 0.735 | 0.876 | 0.289 | 0.535 | 0.116 |
| D_ccmt | `46b56e8e` | 0.995 | 0.983 | 0.987 | 0.711 | 0.882 | 0.242 | 0.417 | 0.150 |
| E_convnext | `96f8b511` | 0.995 | 0.983 | 0.970 | 0.734 | 0.866 | 0.271 | 0.400 | 0.143 |

## Critères de sortie (§6)

- **A_baseline** : ✅ F1 sante in-domain ≥ 0,97 (1.000) ; ✅ rappel « anormal » PlantDoc ≥ 0,80 (0.869)
- **B_strong_aug** : ✅ F1 sante in-domain ≥ 0,97 (1.000) ; ✅ rappel « anormal » PlantDoc ≥ 0,80 (0.960)
- **C_background** : ✅ F1 sante in-domain ≥ 0,97 (1.000) ; ✅ rappel « anormal » PlantDoc ≥ 0,80 (0.973)
- **D_ccmt** : ✅ F1 sante in-domain ≥ 0,97 (0.995) ; ✅ rappel « anormal » PlantDoc ≥ 0,80 (0.987)
- **E_convnext** : ✅ F1 sante in-domain ≥ 0,97 (0.995) ; ✅ rappel « anormal » PlantDoc ≥ 0,80 (0.970)

Détail par run (matrices de confusion, planches Grad-CAM) : `reports/<run_id>/` (non versionné ; `make harvest` depuis Colab).
